# Heisenberg Gate POC: Small Kagome Patch

This notebook tests the proof-of-concept hypothesis on a small open-boundary Kagome patch:

> Adding variational Heisenberg interaction layers improves the variational state toward the finite-size Kagome antiferromagnetic ground state.

The default Hamiltonian uses the physical spin-1/2 convention, `H = J/4 sum(XX + YY + ZZ)`.

In [ ]:
from pathlib import Path
import csv
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / "src"))

from kagome_heisenberg_poc import (
    bond_correlations,
    build_heisenberg_ansatz,
    kagome_patch,
    run_depth_sweep,
    spin_z_expectations,
    statevector_from_params,
)

plt.rcParams["figure.dpi"] = 130

## 1. Build a 6-site Kagome debugging patch

In [ ]:
positions, bonds = kagome_patch(nx=2, ny=1)
num_sites = len(positions)

print(f"sites: {num_sites}")
print(f"bonds: {len(bonds)}")
print(bonds)

fig, ax = plt.subplots(figsize=(5, 2.4))
for i, j in bonds:
    ax.plot(
        [positions[i, 0], positions[j, 0]],
        [positions[i, 1], positions[j, 1]],
        color="0.55",
        lw=2,
        zorder=1,
    )
ax.scatter(positions[:, 0], positions[:, 1], s=120, color="#2563eb", zorder=2)
for site, (x, y) in enumerate(positions):
    ax.text(x, y + 0.08, str(site), ha="center", va="bottom")
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Open 6-site Kagome patch")
plt.show()

## 2. Exact diagonalization and Heisenberg-layer VQE

The ansatz layer applies `exp[-i theta_ij (XX + YY + ZZ)]` on each Kagome bond. This notebook uses bond-dependent parameters because the single shared-parameter version can be too constrained on this tiny open patch.

In [ ]:
MAX_LAYERS = 3
MAXITER = 250
PARAMETERIZATION = "bond"

results, hamiltonian, exact_state = run_depth_sweep(
    num_qubits=num_sites,
    bonds=bonds,
    max_layers=MAX_LAYERS,
    parameterization=PARAMETERIZATION,
    initial_state="singlet_pairs",
    pauli_scale=0.25,
    maxiter=MAXITER,
    seed=11,
)

rows = []
for result in results:
    rows.append(
        {
            "p": result.layers,
            "energy": result.energy,
            "exact": result.exact_energy,
            "error": result.error,
            "fidelity": result.fidelity,
            "entropy": result.entropy,
            "evaluations": result.evaluations,
        }
    )

for row in rows:
    print(
        "p={p}: E={energy:.8f}, error={error:.8f}, fidelity={fidelity:.6f}, "
        "entropy={entropy:.4f}, evals={evaluations}".format(**row)
    )

## 3. Energy and fidelity versus depth

In [ ]:
layers = np.array([result.layers for result in results])
energies = np.array([result.energy for result in results])
fidelities = np.array([result.fidelity or np.nan for result in results])
exact_energy = results[0].exact_energy

fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.2))
axes[0].plot(layers, energies, marker="o")
axes[0].axhline(exact_energy, color="black", ls="--", lw=1, label="exact")
axes[0].set_xlabel("Heisenberg layers p")
axes[0].set_ylabel("Energy")
axes[0].legend()

axes[1].plot(layers, fidelities, marker="o", color="#16a34a")
axes[1].set_xlabel("Heisenberg layers p")
axes[1].set_ylabel("Fidelity with exact ground state")
axes[1].set_ylim(-0.03, 1.03)

fig.tight_layout()
plt.show()

## 4. Local magnetization and bond-energy diagnostics

In [ ]:
best = min(results, key=lambda item: item.energy)
best_circuit, best_parameters = build_heisenberg_ansatz(
    num_sites,
    bonds,
    layers=best.layers,
    parameterization=PARAMETERIZATION,
    initial_state="singlet_pairs",
)
best_state = statevector_from_params(best_circuit, best_parameters, best.parameters)

sz = spin_z_expectations(best_state)
correlations = bond_correlations(best_state, num_sites, bonds, pauli_scale=0.25)

print("<S_i^z>:", np.round(sz, 6))
print("bond <S_i . S_j>:")
for bond, value in correlations.items():
    print(f"  {bond}: {value:.8f}")

values = np.array(list(correlations.values()))
norm = plt.Normalize(values.min(), values.max())
cmap = plt.cm.viridis_r

fig, ax = plt.subplots(figsize=(5, 2.4))
for i, j in bonds:
    value = correlations[(i, j)]
    ax.plot(
        [positions[i, 0], positions[j, 0]],
        [positions[i, 1], positions[j, 1]],
        color=cmap(norm(value)),
        lw=5,
        solid_capstyle="round",
        zorder=1,
    )
ax.scatter(positions[:, 0], positions[:, 1], s=110, color="white", edgecolor="black", zorder=2)
for site, (x, y) in enumerate(positions):
    ax.text(x, y, str(site), ha="center", va="center", fontsize=8, zorder=3)
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
fig.colorbar(sm, ax=ax, label="<S_i . S_j>")
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"Best depth p={best.layers}")
plt.show()

## 5. Save numerical results

In [ ]:
output_dir = PROJECT / "results"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "small_patch_depth_sweep.csv"

with output_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print(output_path)